# SCADA Data Exploration and Prototyping

This notebook explores the `sample_scada.csv` dataset to understand its structure and prototype visualizations for the Streamlit dashboard.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Load data
df = pd.read_csv('sample_scada.csv')
df['date'] = pd.to_datetime(df['date'],format = "%Y%m%d")
df.head()

,date,block,Raw_Frequency,MP_Schedule,MP_Drawal,MP_Demand,CZ_Total_Schedule,CZ_Demand,EZ_Total_Schedule,EZ_Demand,...,Total_IPP_Gen,Solar,Wind,CZ_Demand_Avg,EZ_Demand_Avg,WZ_Demand_Avg,MP_Demand_Avg,ISP_Hydro,OSP_Hydro,date_int
0,2025-11-01,1,50.011160,4747.67,4659.50,7061.47,1629.54,2327.91,1416.93,1941.00,...,330.98,0.0,91.52,2327.91,1941.00,2832.16,7061.47,0.626415,286.97,20251101
1,2025-11-01,2,49.964285,4592.07,5017.17,7099.24,1511.08,2158.68,1454.39,1992.31,...,375.35,0.0,31.22,2158.68,1992.31,2827.42,7099.24,0.503718,300.55,20251101
2,2025-11-01,3,49.967981,5462.07,4637.31,7404.14,1673.58,2390.83,1540.46,2110.22,...,370.61,0.0,26.56,2390.83,2110.22,2907.63,7404.14,0.613110,337.07,20251101
3,2025-11-01,4,49.890167,4856.33,5882.78,7271.72,1667.50,2382.15,1502.47,2058.18,...,346.02,0.0,26.68,2382.15,2058.18,2853.87,7271.72,0.483235,255.13,20251101
4,2025-11-01,5,50.001945,5340.65,4807.27,7454.27,1661.48,2373.54,1542.47,2112.97,...,374.83,0.0,119.66,2373.54,2112.97,2975.86,7454.27,0.561705,260.35,20251101


## 1. Key Metric Trend Lines

Let's visualize the demand trends over time.

In [2]:
# Daily Total Demand Trend
daily_demand = df.groupby('date')['MP_Drawal'].sum().reset_index()

fig_demand_trend = px.line(
    daily_demand, 
    x='date', 
    y='MP_Drawal', 
    title='Total Daily Demand Energy Over Time',
    labels={'MP_Drawal': 'Total Demand (MWh)', 'date': 'Date'}
)
fig_demand_trend.update_layout(template='plotly_white')
fig_demand_trend.show()

In [3]:
# Peak vs Min vs Avg Demand per block
# Aggregate per day to simplify
daily_stats = df.groupby('date').agg({
    'peak_demand': 'max',
    'min_demand': 'min',
    'avg_demand': 'mean'
}).reset_index()

fig_demand_stats = go.Figure()
fig_demand_stats.add_trace(go.Scatter(x=daily_stats['date'], y=daily_stats['peak_demand'], mode='lines', name='Peak Demand'))
fig_demand_stats.add_trace(go.Scatter(x=daily_stats['date'], y=daily_stats['avg_demand'], mode='lines', name='Avg Demand'))
fig_demand_stats.add_trace(go.Scatter(x=daily_stats['date'], y=daily_stats['min_demand'], mode='lines', name='Min Demand'))

fig_demand_stats.update_layout(
    title='Daily Peak, Min, and Average Demand',
    xaxis_title='Date',
    yaxis_title='Demand (MW)',
    template='plotly_white',
    hovermode='x unified'
)
fig_demand_stats.show()

KeyError: "Column(s) ['avg_demand', 'min_demand', 'peak_demand'] do not exist"

## 2. Distribution Plots

Understanding the distribution of demand across different regions.

In [ ]:
# Box plot for regional demand distribution
region_cols = ['WRP', 'NRP', 'ERP', 'SRP', 'NERP']
# Melt the dataframe to have regions in one column and values in another
df_regions = df.melt(id_vars=['date', 'block_no'], value_vars=region_cols, var_name='Region', value_name='Demand')

fig_regional_dist = px.box(
    df_regions, 
    x='Region', 
    y='Demand', 
    title='Demand Distribution by Region',
    color='Region',
    template='plotly_white'
)
fig_regional_dist.show()

## 3. Comparison Charts (Generation Mix)

Looking at the proportion of different generation types.

In [ ]:
# Daily generation mix (stacked area chart)
gen_cols = ['thermal_gen', 'hydel_gen', 'renewable_gen']
daily_gen = df.groupby('date')[gen_cols].sum().reset_index()

fig_gen_mix = go.Figure()
for col in gen_cols:
    fig_gen_mix.add_trace(go.Scatter(
        x=daily_gen['date'], 
        y=daily_gen[col],
        mode='lines',
        name=col.replace('_', ' ').title(),
        stackgroup='one' # This creates the stacked area effect
    ))

fig_gen_mix.update_layout(
    title='Total Generation Mix Over Time',
    xaxis_title='Date',
    yaxis_title='Generation (MWh)',
    template='plotly_white',
    hovermode='x unified'
)
fig_gen_mix.show()

## 4. Time Profiles (Intraday)

How does demand change throughout the 96 blocks of a day?

In [ ]:
# Average Demand Profile by Block
avg_block_profile = df.groupby('block_no')['avg_demand'].mean().reset_index()

fig_intraday = px.line(
    avg_block_profile, 
    x='block_no', 
    y='avg_demand', 
    title='Average Intraday Demand Profile',
    labels={'avg_demand': 'Average Demand (MW)', 'block_no': 'Time Block (1-96)'}
)
fig_intraday.update_layout(template='plotly_white')
fig_intraday.show()